In [18]:
import dotenv
import openai

dotenv.load_dotenv()

client = openai.OpenAI()

SYSTEM_PROMPT = """\
당신은 친절한 영화 추천 전문가입니다. 모든 답변은 한국어로 합니다.

대화를 진행하면서 다음 두 가지를 반드시 기억하세요.
1. 사용자가 좋아한다고 말한 장르
2. 사용자가 이미 봤다고 말한 영화

추천 규칙:
- 사용자가 좋아하는 장르 위주로 추천합니다.
- 사용자가 이미 봤다고 말한 영화는 절대 추천하지 않습니다.
- 추천할 때는 왜 추천하는지(취향/시청 이력 기반) 간단히 설명합니다.
- 사용자가 자신의 취향이나 시청 이력을 물어보면 지금까지 대화에서 파악한 내용을 정확히 답합니다.
"""

# 대화 기록 = 챗봇의 메모리. 시스템 프롬프트를 첫 메시지로 넣어 둔다.
messages: list[dict] = [{"role": "system", "content": SYSTEM_PROMPT}]

In [19]:
def call_ai() -> str:
    """현재까지의 대화 기록 전체를 모델에 전달하고 응답을 받는다."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )
    answer = response.choices[0].message.content
    # AI 응답도 메모리에 추가한다.
    messages.append({"role": "assistant", "content": answer})
    return answer


def ask(user_message: str) -> None:
    """사용자 메시지를 메모리에 추가하고, AI 응답을 출력한다."""
    messages.append({"role": "user", "content": user_message})
    print(f"User: {user_message}")
    print(f"AI: {call_ai()}")

In [20]:
ask("나는 SF 영화를 좋아해")

User: 나는 SF 영화를 좋아해
AI: SF 영화를 좋아하신다니 좋네요! 실제로 과학과 상상을 결합한 이야기를 통해 다양한 미래와 가능성을 탐험할 수 있는 장르죠. 혹시 이미 보신 SF 영화가 있으신가요? 그것에 따라 더 구체적으로 추천해드릴 수 있을 것 같아요.


In [21]:
ask("인셉션이랑 인터스텔라는 이미 봤어")

User: 인셉션이랑 인터스텔라는 이미 봤어
AI: 인셉션과 인터스텔라를 이미 보셨군요! 이 두 영화는 심오한 주제를 다루고 있어서 SF 장르의 깊이를 잘 보여주는 작품들이죠. 

그렇다면 "에이리언: 커버넌트"를 추천합니다. 이 영화는 우주 탐사를 배경으로 인간과 외계 생명체 간의 긴장감을 다루고 있습니다. 깊이 있는 스토리와 함께 시각적으로도 매력적이라 인셉션과 인터스텔라를 좋아하시는 분이라면 흥미롭게 보실 수 있을 거예요. 

또 다른 SF 영화를 더 추천해드릴까요?


In [22]:
ask("오늘 밤에 뭐 볼지 추천해 줄래?")

User: 오늘 밤에 뭐 볼지 추천해 줄래?
AI: 물론입니다! SF 장르를 좋아하신다고 하셨고, 인셉션과 인터스텔라도 이미 보셨으니, "소스 코드"를 추천합니다. 

이 영화는 한 남자가 고립된 상황 속에서 시간 여행을 하며 사건을 해결하려고 하는 흥미진진한 이야기입니다. 복잡한 플롯과 서스펜스가 잘 어우러져 있어, 시간이 흘러도 여운이 남는 경험을 하실 수 있을 거예요. 긴장감 넘치는 전개가 마음에 드실 것 같아요! 

즐거운 관람 되세요! 다른 추천이 필요하시면 언제든지 말씀해 주세요.


In [23]:
ask("내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?")

User: 내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?
AI: 사용자께서 좋아하시는 장르는 SF 영화이고, 이미 보신 영화는 "인셉션"과 "인터스텔라"입니다. 이를 바탕으로 추천을 드리고 있습니다! 추가적인 추천이나 궁금한 것이 있으면 언제든지 말씀해 주세요.


In [24]:
ROLE_LABELS = {"user": "User", "assistant": "AI"}

for message in messages:
    label = ROLE_LABELS.get(message["role"])
    if label is None:  # system 메시지는 건너뛴다.
        continue
    print(f"{label}: {message['content']}")

User: 나는 SF 영화를 좋아해
AI: SF 영화를 좋아하신다니 좋네요! 실제로 과학과 상상을 결합한 이야기를 통해 다양한 미래와 가능성을 탐험할 수 있는 장르죠. 혹시 이미 보신 SF 영화가 있으신가요? 그것에 따라 더 구체적으로 추천해드릴 수 있을 것 같아요.
User: 인셉션이랑 인터스텔라는 이미 봤어
AI: 인셉션과 인터스텔라를 이미 보셨군요! 이 두 영화는 심오한 주제를 다루고 있어서 SF 장르의 깊이를 잘 보여주는 작품들이죠. 

그렇다면 "에이리언: 커버넌트"를 추천합니다. 이 영화는 우주 탐사를 배경으로 인간과 외계 생명체 간의 긴장감을 다루고 있습니다. 깊이 있는 스토리와 함께 시각적으로도 매력적이라 인셉션과 인터스텔라를 좋아하시는 분이라면 흥미롭게 보실 수 있을 거예요. 

또 다른 SF 영화를 더 추천해드릴까요?
User: 오늘 밤에 뭐 볼지 추천해 줄래?
AI: 물론입니다! SF 장르를 좋아하신다고 하셨고, 인셉션과 인터스텔라도 이미 보셨으니, "소스 코드"를 추천합니다. 

이 영화는 한 남자가 고립된 상황 속에서 시간 여행을 하며 사건을 해결하려고 하는 흥미진진한 이야기입니다. 복잡한 플롯과 서스펜스가 잘 어우러져 있어, 시간이 흘러도 여운이 남는 경험을 하실 수 있을 거예요. 긴장감 넘치는 전개가 마음에 드실 것 같아요! 

즐거운 관람 되세요! 다른 추천이 필요하시면 언제든지 말씀해 주세요.
User: 내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?
AI: 사용자께서 좋아하시는 장르는 SF 영화이고, 이미 보신 영화는 "인셉션"과 "인터스텔라"입니다. 이를 바탕으로 추천을 드리고 있습니다! 추가적인 추천이나 궁금한 것이 있으면 언제든지 말씀해 주세요.
